# HTF Policy Comparison

Reproducible viewer for the HTF synchronization-policy simulation.

The heavy lifting lives in `_htf_policy_simulation.py`. Re-run that
script whenever the window, basket, or policy set changes, then
re-execute this notebook to refresh the visualizations and tables.

**Policy index**
- `A`     — broker-only HTF (no synth). Strictest causality.
- `B`     — current production: always extend HTF with synth from M5.
- `C_15`  — hybrid: synth only if last broker H1 closed > 15 min ago.
- `C_60`  — hybrid: synth only if last broker H1 closed > 60 min ago.
- `C_120` — hybrid: synth only if last broker H1 closed > 120 min ago.
- `D_2`   — broker-only + require last 2 H1 bars same trend.
- `D_3`   — broker-only + require last 3 H1 bars same trend.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SIM  = ROOT / 'notebooks' / 'data' / 'htf_policy'
assert SIM.exists(), 'Run notebooks/_htf_policy_simulation.py first.'

metrics = pd.read_csv(SIM / 'metrics.csv')
signals = pd.read_csv(SIM / 'signals.csv') if (SIM / 'signals.csv').exists() else pd.DataFrame()
trades  = pd.read_csv(SIM / 'trades.csv')  if (SIM / 'trades.csv').exists()  else pd.DataFrame()
diags   = pd.read_csv(SIM / 'diags.csv')   if (SIM / 'diags.csv').exists()   else pd.DataFrame()

print(f'metrics: {len(metrics)} rows  |  signals: {len(signals)}  |  trades: {len(trades)}  |  diags: {len(diags)}')
metrics.head(12)

metrics: 7 rows  |  signals: 45  |  trades: 8  |  diags: 2016


,trades,WR,PF,expectancy_R,sum_R,net_$,max_dd_R,recovery_factor,avg_RR,sharpe_R,symbol,policy,signal_count,elapsed_s
0,0,0.0,0.000,0.0000,0.000,0.0,0.000,0.000,0.000,0.000,GBPUSD,A,3,14.6
1,2,50.0,1.051,0.0138,0.028,-2.2,0.543,0.051,1.051,0.025,GBPUSD,B,9,16.2
2,2,50.0,1.051,0.0138,0.028,-2.2,0.543,0.051,1.051,0.025,GBPUSD,C_15,9,18.3
3,2,50.0,1.051,0.0138,0.028,-2.2,0.543,0.051,1.051,0.025,GBPUSD,C_60,9,15.6
4,2,50.0,1.051,0.0138,0.028,-2.2,0.543,0.051,1.051,0.025,GBPUSD,C_120,9,15.0
5,0,0.0,0.000,0.0000,0.000,0.0,0.000,0.000,0.000,0.000,GBPUSD,D_2,3,16.0
6,0,0.0,0.000,0.0000,0.000,0.0,0.000,0.000,0.000,0.000,GBPUSD,D_3,3,17.7


## 1. Portfolio summary per policy

In [2]:
agg = (metrics.groupby('policy', as_index=False)
              .agg(signals=('signal_count','sum'),
                   trades=('trades','sum'),
                   sum_R=('sum_R','sum'),
                   net_=('net_$','sum')))
agg['signals_per_trade'] = (agg['signals']/agg['trades']).round(2)
agg['cascade_loss_%'] = ((1 - agg['trades']/agg['signals'])*100).round(1)
policy_order = ['A','B','C_15','C_60','C_120','D_2','D_3']
agg = agg.set_index('policy').reindex([p for p in policy_order if p in agg.index]).reset_index()
agg

,policy,signals,trades,sum_R,net_,signals_per_trade,cascade_loss_%


## 2. Per-symbol breakdown

In [3]:
pivot_trades = metrics.pivot(index='symbol', columns='policy', values='trades').reindex(columns=policy_order, fill_value=0).fillna(0).astype(int)
pivot_R      = metrics.pivot(index='symbol', columns='policy', values='sum_R').reindex(columns=policy_order, fill_value=0).fillna(0).round(2)
pivot_WR     = metrics.pivot(index='symbol', columns='policy', values='WR').reindex(columns=policy_order, fill_value=0).fillna(0).round(1)
pivot_DD     = metrics.pivot(index='symbol', columns='policy', values='max_dd_R').reindex(columns=policy_order, fill_value=0).fillna(0).round(2)
print('Trades per symbol per policy'); print(pivot_trades.to_string())
print('\nSum R per symbol per policy');   print(pivot_R.to_string())
print('\nWR% per symbol per policy');     print(pivot_WR.to_string())
print('\nMax DD (R) per symbol per policy'); print(pivot_DD.to_string())

Trades per symbol per policy
policy  A  B  C_15  C_60  C_120  D_2  D_3
symbol                                   
GBPUSD  0  2     2     2      2    0    0

Sum R per symbol per policy
policy    A     B  C_15  C_60  C_120  D_2  D_3
symbol                                        
GBPUSD  0.0  0.03  0.03  0.03   0.03  0.0  0.0

WR% per symbol per policy
policy    A     B  C_15  C_60  C_120  D_2  D_3
symbol                                        
GBPUSD  0.0  50.0  50.0  50.0   50.0  0.0  0.0

Max DD (R) per symbol per policy
policy    A     B  C_15  C_60  C_120  D_2  D_3
symbol                                        
GBPUSD  0.0  0.54  0.54  0.54   0.54  0.0  0.0


## 3. Signal-set Jaccard (which policies fire on the same bars)

In [4]:
def policy_keys(p):
    s = signals[signals['policy']==p]
    return set(zip(s['symbol'], s['bar_time'], s['direction']))

polset = {p: policy_keys(p) for p in policy_order if p in set(signals['policy'].unique())}
jaccard = pd.DataFrame(index=polset.keys(), columns=polset.keys(), dtype=float)
for a in polset:
    for b in polset:
        u = polset[a] | polset[b]; i = polset[a] & polset[b]
        jaccard.loc[a, b] = round(len(i)/len(u), 3) if u else 1.0
print('Signal-set Jaccard similarity (1 = identical):')
jaccard

Signal-set Jaccard similarity (1 = identical):


,A,B,C_15,C_60,C_120,D_2,D_3
A,1.000,0.333,0.333,0.333,0.333,1.000,1.000
B,0.333,1.000,1.000,1.000,1.000,0.333,0.333
C_15,0.333,1.000,1.000,1.000,1.000,0.333,0.333
C_60,0.333,1.000,1.000,1.000,1.000,0.333,0.333
C_120,0.333,1.000,1.000,1.000,1.000,0.333,0.333
D_2,1.000,0.333,0.333,0.333,0.333,1.000,1.000
D_3,1.000,0.333,0.333,0.333,0.333,1.000,1.000


## 4. Visualizations

Run `python notebooks/_htf_policy_visualize.py` to regenerate the PNGs.

In [5]:
from IPython.display import Image
FIG = SIM / 'figs'
for p in sorted(FIG.glob('*.png')):
    print(p.name)